# 第 10 章 · 评估与测试工程

> **你已会**：Agent、工具、回调、会话、事件流。
>
> **读完后你能**：① 读懂 ADK 评估集的结构；② 用质量脚本或 `adk eval` 回放评估；③ 把指标变成可执行的发布门禁；④ 区分"离线单元测试"和"在线评估"。
>
> **本章不碰**：模型微调、LangSmith 等外部平台。

**怎么读**

- **主线（约 45 分钟）**：§1 为什么评估 → §2 评估集结构 → §4 质量门禁。
- **深读**：§3 命令行回放、§5 两层测试。
- 本章的代码默认**离线**（读评估集与报告）；真实回放需要 `DEEPSEEK_API_KEY`，命令在 §3。

---

## 1. 运行正常，不等于回答正确

回调能告诉你"没有崩"，日志能告诉你"调了哪个工具"，但它们都不能回答：

- 该调的工具调了吗？顺序对吗？
- 最终答案完成任务了吗？
- 提示注入被挡住了吗？
- 成本、延迟还守得住吗？

评估的做法是**用一组固定的交互剧本回放 Agent，再对轨迹和回答打分**：

```mermaid
flowchart LR
    D["评估集<br/>用户说什么 → 期望调什么 → 期望怎么答"] --> R["Runner 回放"]
    R --> A["Agent"]
    A --> M["指标：轨迹 / 回答 / 格式 / 安全 / 延迟 / 成本"]
    M --> G{"质量门禁"}
    G -->|通过| P["允许发布"]
    G -->|失败| X["定位失败案例 → 修改 → 重跑"]
```

> 📌 **评估集 = 期望的交互剧本。** 先写评估集，再改 prompt。否则你永远在"感觉好像变好了"。

**先猜：一个 26 条的评估集，如果只跑 1 条且全部通过，能不能据此发布？**

## 2. 评估集长什么样

本仓库的 [`eval_example/eval_sets/after_sales.evalset.json`](eval_example/eval_sets/after_sales.evalset.json) 是一份 ADK 原生评估集：

| 字段 | 作用 |
|---|---|
| `eval_set_id` | 数据集标识，报告和基线都靠它对齐 |
| `eval_cases[]` | 一条案例 |
| `conversation[]` | 多轮对话：`user_content`、`final_response`、`intermediate_data.tool_uses` |
| `test_spec.category` | 案例分类：normal / edge / safety / multi_turn… |
| `test_spec.turns[]` | 每轮的必含关键词、禁用词、格式标记 |

先把它读进来，看看规模与覆盖面：

In [1]:
import json
from collections import Counter
from pathlib import Path

root = Path.cwd()
if not (root / "eval_example").exists():
    root = root / "google-adk"

dataset_path = root / "eval_example" / "eval_sets" / "after_sales.evalset.json"
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))

print("数据集：", dataset["eval_set_id"])
print("案例数：", len(dataset["eval_cases"]))

categories = Counter(
    case["test_spec"]["category"] for case in dataset["eval_cases"]
)
print("分类分布：", dict(categories))

first = dataset["eval_cases"][0]
print("\n第一条案例：", first["eval_id"])
print("用户输入：", first["conversation"][0]["user_content"]["parts"][0]["text"])
print("测试规格：", first["test_spec"])

assert len(dataset["eval_cases"]) >= 20
assert {"normal", "edge", "safety", "multi_turn"} <= set(categories)
print("✅ 规模与最小覆盖面符合要求")

数据集： after_sales_regression_v1
案例数： 26
分类分布： {'normal': 8, 'edge': 6, 'tool_failure': 2, 'high_risk': 3, 'safety': 3, 'safety_negative': 1, 'multi_turn': 2, 'multi_tool': 1}

第一条案例： order_delivered
用户输入： 查询订单 A1001 的状态。
测试规格： {'category': 'normal', 'turns': [{'required_keywords': ['A1001', '已签收'], 'required_markers': ['处理结果：', '下一步：']}]}
✅ 规模与最小覆盖面符合要求


### 2.1 一条好案例包含什么

| 要素 | 反例 | 正例 |
|---|---|---|
| 用户输入 | "帮我处理一下" | "订单 A1001 已签收，想退款 299 元" |
| 期望工具 | 不写 | `query_order → assess_refund_eligibility` |
| 期望结果 | 只写"回答正确" | 必须包含 `A1001`、`已签收`；禁止承诺退款 |
| 分类 | 不分类 | `high_risk`，单独统计通过率 |
| 轮次 | 只测单轮 | 第二轮用"它还能退吗"验证指代 |

> ⚠️ **只断言文案相似度的评估很脆弱。** 把"工具轨迹""必含字段""禁止词"和"格式标记"一起用，才能区分"答得不一样"和"答错了"。

---

## 3. 回放：两条路径

### 3.1 本仓库可运行的路径

由于 `google-adk[eval]` 会间接引入与当前 LiteLLM 版本冲突的依赖，本仓库没有安装完整评估扩展，而是用 [`eval_example/run_quality_eval.py`](eval_example/run_quality_eval.py) 实现同样的轨迹语义与 ROUGE 指标。

在 `google-adk/` 目录下，先做**离线校验**（不调模型）：

```bash
uv run --project .. python -m eval_example.run_quality_eval --dry-run
```

再跑**单条冒烟**（会调模型）：

```bash
uv run --project .. --env-file ../.env python -m eval_example.run_quality_eval --case order_delivered
```

```bash
# 跑全集（会消耗较多模型调用）
uv run --project .. --env-file ../.env python -m eval_example.run_quality_eval
```

### 3.2 官方 `adk eval`（需隔离环境）

如果能安装完整评估扩展，可以使用官方命令：

```bash
uv add "google-adk[eval]"
adk eval eval_example eval_example/eval_sets/after_sales.evalset.json:order_delivered   --config_file_path eval_example/evaluation_config.json   --print_detailed_results
```

> ⚠️ **不要在锁定了 LiteLLM 版本的主仓库里强行安装 `[eval]` extra**，否则会出现依赖降级冲突。先在隔离环境验证。

阈值写在 [`eval_example/evaluation_config.json`](eval_example/evaluation_config.json)：

| 指标 | 含义 | 本例阈值 |
|---|---|---|
| `tool_trajectory_avg_score` | 工具调用序列与期望的匹配度 | 1.0 |
| `response_match_score` | 最终回答与参考答案的相似度 | 0.45 |

> 📌 **轨迹指标优先于文案指标。** 工具调错而文案相似，是更危险的失败；轨迹阈值应当先收紧。

---

## 4. 质量门禁：把"感觉"变成退出码

`adk eval` 负责轨迹和相似度；[`eval_example/run_quality_eval.py`](eval_example/run_quality_eval.py) 在此基础上补了任务成功率、格式、安全、延迟和成本门禁，并用**退出码**表达结果。

读一份真实报告，看看门禁怎么判：

In [2]:
report_path = root / "eval_example" / "reports" / "latest.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

print("模型：", report["model"])
print("数据集：", report["dataset_id"])
print("指标：")
for key, value in report["metrics"].items():
    print(f"  {key}: {value}")

failed = [name for name, ok in report["gates"].items() if not ok]
print("失败门禁：", failed)

# 这份样例只跑了 1 条，因此数据集规模门禁必然失败。
# 这正是"单条通过 ≠ 可以发布"的直观证据。
assert "dataset_min_cases" in failed
assert report["passed"] is False
print("✅ 门禁如实拦住了'只跑一条就发布'")

模型： deepseek/deepseek-chat
数据集： after_sales_regression_v1
指标：
  dataset_case_count: 1
  turn_count: 1
  case_pass_rate: 1.0
  task_success_rate: 1.0
  tool_selection_accuracy: 1.0
  response_match_score: 0.5148514851485148
  format_compliance_rate: 1.0
  safety_case_pass_rate: 1.0
  p95_latency_ms: 3969.02
  input_tokens: 2451
  output_tokens: 90
  estimated_total_cost_usd: 0.000761
失败门禁： ['dataset_min_cases']
✅ 门禁如实拦住了'只跑一条就发布'


### 4.1 门禁应该拦住什么

| 门禁 | 典型阈值 | 为什么 |
|---|---|---|
| 数据集规模 | ≥ 20 条 | 一条通过没有统计意义 |
| 任务成功率 | ≥ 85% | 业务是否真的完成 |
| 工具选择准确率 | ≥ 85% | 轨迹是否正确 |
| 格式合规率 | ≥ 95% | 下游系统能否解析 |
| 安全用例通过率 | 100% | 安全没有"差不多" |
| P95 延迟 | ≤ 15s | 用户体验 |
| 估算成本 | ≤ 预算 | 防止悄悄涨价 |
| 相对基线回退 | ≤ 5% | 防止"改一个坏三个" |

> ⚠️ **门禁必须能失败。** 一个永远不会红的质量门禁，等于没有门禁。发布前先故意制造一次失败，确认退出码是 1。

---

## 5. 两层测试：离线快、在线真

| 层 | 工具 | 是否调模型 | 什么时候跑 | 抓什么 |
|---|---|---|---|---|
| 离线单元测试 | `pytest` / `unittest` | 否 | 每次提交 | 护栏、幂等、数据契约、schema |
| 在线评估 | `adk eval` / 质量脚本 | 是 | 合并前、发版前 | 轨迹、回答质量、成本 |

仓库里的 [`eval_example/tests/test_project.py`](eval_example/tests/test_project.py) 是离线层：不调模型，验证护栏短路、退款无副作用、工具幂等、数据集结构。

```bash
uv run python -m pytest eval_example/tests -q
```

> 📌 **分层原则**：能用确定性断言证明的，不要花模型调用；只有"模型真的能不能做对"才交给在线评估。

---

## 6. 评估驱动迭代的闭环

1. 从线上失败里挑一条真实请求；
2. 把它写成评估案例，补上期望工具与禁止词；
3. 先跑，确认它能复现失败；
4. 改 prompt / 工具 / 护栏；
5. 重跑全集，确认没有回退；
6. 把基线更新到新版本。

> ⚠️ **不要先改 prompt 再补案例。** 那样你只会写出"刚好能过"的案例，评估就失去了独立性。

---

## 📌 用五句话复述本章

1. **运行正常不等于回答正确**；评估集是期望的交互剧本。
2. **评估集要覆盖正常、边界、安全、多轮**，并标注期望工具与禁用词。
3. **轨迹指标优先于文案相似度**；回放负责产生指标，质量脚本负责门禁。
4. **门禁必须能失败**，并用退出码进入 CI。
5. **离线测试管确定性，在线评估管模型能力**，两层都要有。

---

## 🧪 本章练习

### 1. 给评估集加一条安全案例（基础）

- [ ] 写一条提示注入或多轮越权案例
- [ ] 标注 `category: safety`
- [ ] 写清必含字段与禁止词
- [ ] 用离线脚本校验 schema

### 2. 设计一组门禁（进阶）

- [ ] 为你的 Agent 选 6 个指标
- [ ] 每个指标给出阈值和理由
- [ ] 说明哪一项必须 100%
- [ ] 故意制造一次失败，确认退出码非 0

### 3. 从失败到回归（工程）

- [ ] 找一条真实失败请求
- [ ] 写成评估案例并复现失败
- [ ] 修改 prompt 或工具后重跑
- [ ] 确认其他案例没有回退

### 4. 两层测试分工（综合）

- [ ] 列出 5 个可以用离线测试覆盖的点
- [ ] 列出 3 个必须在线评估的点
- [ ] 说明为什么不能互换

参考实现：[practice/c10_1.py](practice/c10_1.py)。